# Lenet을 이용한 CNN학습

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torchvision import datasets
import torchvision.transforms as transforms # 데이터 전처리
from tensorboardX import SummaryWriter


In [3]:
data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((32, 32)),
    transforms.Normalize((0.5), (1.0))
])

In [4]:

train_data = datasets.MNIST(root='../../data_resources/', train=True, download=True, transform=data_transform)
test_data = datasets.MNIST(root='../../data_resources/', train=False, download=True, transform=data_transform)


In [5]:
train_data.data[0].shape

torch.Size([28, 28])

## 미니 배치 사이즈로 쪼개기

In [6]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data, batch_size=60, shuffle=True)
test_loader = DataLoader(test_data, batch_size=60)

In [7]:
# iter(train_loader) -> train_loader를 이터레이터로 바꿈
# 미니배치 단위로 데이터를 차례대로 꺼낼 준비를 한다
data, label = next(iter(train_loader))
# next(iter(train_loader))는 다음 배치를 꺼냄 여기서는 첫배치
# (data, label) tuple이 나옴
print(data.shape, label.shape)

torch.Size([60, 1, 32, 32]) torch.Size([60])


In [8]:
class Lenet(nn.Module):
    def __init__(self):
        super(Lenet, self).__init__() ## 복사한 부모도 초기화시키겠다
        self.conv1 = nn.Conv2d(1,   6,  kernel_size=5, stride=1)   # 32->28
        self.conv2 = nn.Conv2d(6,   16, kernel_size=5, stride=1)   # 14->10
        self.conv3 = nn.Conv2d(16, 120, kernel_size=5, stride=1)   # 5->1
        self.fc1 = nn.Linear(in_features=120, out_features=84)
        self.fc2 = nn.Linear(in_features=84, out_features=10)  # 보통 fully connection 부분을 수정해가며 튜닝


    def forward(self, x):
        x = self.conv1(x)
        x = F.tanh(x) #6, 28, 28
        x = F.max_pool2d(x,2,2) #6, 14, 14
        x = self.conv2(x) 
        x = F.tanh(x) # 16 2. 2.
        x = F.max_pool2d(x, 2,2) #16, 5, 5
        x = self.conv3(x) 
        x = F.tanh(x) 
        x = x.view(-1, 120)

        x = self.fc1(x)
        x = F.tanh(x)
        x = self.fc2(x)

        return x

        


model = Lenet()

In [ ]:
# import torch.nn as nn
# import torch.nn.functional as F

# class Lenet(nn.Module):
#     def __init__(self):
#         super(Lenet, self).__init__()
#         self.conv1 = nn.Conv2d(1,   6,  kernel_size=5, stride=1)   # 32->28
#         self.conv2 = nn.Conv2d(6,   16, kernel_size=5, stride=1)   # 14->10
#         self.conv3 = nn.Conv2d(16, 120, kernel_size=5, stride=1)   # 5->1
#         self.fc1   = nn.Linear(120, 84)
#         self.fc2   = nn.Linear(84,  10)

#     def forward(self, x):
#         x = torch.tanh(self.conv1(x))             # (N,6,28,28)
#         x = F.max_pool2d(x, 2, 2)                 # (N,6,14,14)
#         x = torch.tanh(self.conv2(x))             # (N,16,10,10)
#         x = F.max_pool2d(x, 2, 2)                 # (N,16,5,5)
#         x = torch.tanh(self.conv3(x))             # (N,120,1,1)
#         x = torch.flatten(x, 1)                   # (N,120)
#         x = torch.tanh(self.fc1(x))               # (N,84)
#         x = self.fc2(x)                           # (N,10)  ← 마지막 활성화 없음 (CELoss용)
#         return x
    
# model = Lenet()

In [9]:
from torchsummary import summary
summary(model, input_size=(1, 32, 32))



Layer (type:depth-idx)                   Param #
├─Conv2d: 1-1                            156
├─Conv2d: 1-2                            2,416
├─Conv2d: 1-3                            48,120
├─Linear: 1-4                            10,164
├─Linear: 1-5                            850
Total params: 61,706
Trainable params: 61,706
Non-trainable params: 0


Layer (type:depth-idx)                   Param #
├─Conv2d: 1-1                            156
├─Conv2d: 1-2                            2,416
├─Conv2d: 1-3                            48,120
├─Linear: 1-4                            10,164
├─Linear: 1-5                            850
Total params: 61,706
Trainable params: 61,706
Non-trainable params: 0

In [10]:
from tensorboardX import SummaryWriter
writer = SummaryWriter()

lr = 1e-3
criterion = nn.CrossEntropyLoss()
optim = Adam(model.parameters(), lr=lr)
epochs = 3

# 모델을 gpu로
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

step = 0
for epoch in range(epochs):
    model.train()

    for data, label in train_loader:
        optim.zero_grad()
        
        pred = model(data.to(device)) #[32, 1, 32, 32]

        loss = criterion(pred, label.to(device))

        writer.add_scalar('Loss/train', loss.item(), step)
        step += 1

        loss.backward()
        optim.step()
    print(f"{epoch+1} loss value: {loss.item()}")



1 loss value: 0.10275950282812119
2 loss value: 0.042741719633340836
3 loss value: 0.0077249398455023766


In [11]:
#모델 평가 코드작성

model.eval() # 평가 모드로 전환
correct = 0
total = 0

with torch.no_grad(): # gradient 계산 비활성화
    for images, labels in test_loader:
        images = images.to(device)  # (B, 1, 32, 32) 이어야 함
        labels = labels.to(device)

        logits = model(images)
        pred = logits.argmax(dim=1)
        correct += (pred == labels).sum().item()
        total += label.size(0)

accuracy = correct / total
print(f"정확도: {accuracy:4f}")


정확도: 0.983633


In [ ]:
#수업자료 - 데이터 - 4.jpg 

from PIL import Image, ImageOps

img = Image.open("../../data_resources/4.jpg")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.eval()

#이미지 전처리
infer_transform = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor(),
    transforms.Grayscale(),
    transforms.Normalize((0.5), (1.0))

])

infer_img = infer_transform(img)

infer_img.size()

torch.Size([1, 32, 32])

In [15]:
model(infer_img.to(device)).argmax(dim=1).item()

4